# Genixcure — 10-sample PCR merged unique counts (fastp, optimized)

**Analysis output:** one `<sample>_merged_unique_sequences.csv.gz` per sample, fields `sequence,read_count,length_bp`, plus status/inventory tables. Fully identical merged sequences are aggregated **without primer/index trimming**. No similarity clustering.

**Kept unchanged** fastp settings: `--merge --disable_adapter_trimming --disable_quality_filtering --disable_length_filtering --disable_trim_poly_g --overlap_len_require 20 --overlap_diff_limit 5 --overlap_diff_percent_limit 20`. Same counting semantics as the previous notebook.

**Speed change:** download R1/R2 concurrently to `/content` local disk, use a **named FIFO** between fastp and `sort | uniq` to avoid writing and rereading an intermediate merged FASTQ; only final `.csv.gz` goes to Drive. Samples are sequential to limit disk/RAM. Can set `USE_FIFO=False` for fallback if the runtime has a FIFO compatibility problem. No incomplete output is marked finished. `OVERWRITE=False` skips existing verified outputs.

Original Drive data are not modified. Remote FASTQ must still pass through the network; Drive mount access is not equivalent to local SSD I/O.


In [ ]:
!which fastp || (sudo apt-get update -qq >/dev/null && sudo apt-get install -y -qq fastp >/dev/null)
!fastp --version


In [ ]:
from google.colab import auth, drive
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
from concurrent.futures import ThreadPoolExecutor
import google.auth
import os, re, csv, gzip, json, shutil, subprocess, time, hashlib, glob
from pathlib import Path
import pandas as pd

auth.authenticate_user()
drive.mount('/content/drive')
creds, _ = google.auth.default(scopes=['https://www.googleapis.com/auth/drive.readonly'])
service = build('drive','v3',credentials=creds,cache_discovery=False)

FOLDER_IDS = {
    'Brain_1':'1DlWiq9dpcvZZ6Xpw8uovd8EhTxTIKRaZ',
    'Brain_2':'1uDkNHE_t8Fs6lXtd-Ftbi1SxaOhvK4mf',
    'Brain_3':'1woHxLMEJYnw_nF_sE0lVHqhqvTN-fAVq',
    'Motor_1':'1yPdiJB5jnAN_lqdrPGUpALbsXm7Q2Jv9',
    'Motor_2':'1brTJvspA9Q-RSjPkuEkfoPqu685_9hmL',
    'Motor_3':'1tEJLs5pyPrV8nvwH86XGDSWRcUb6YWdJ',
    'SH_SY5Y_1':'1mc0tlqGQaiveM3OBpPof__5MoK0EAZfv',
    'SH_SY5Y_2':'1sMzD62LwPBFFuHvFvC2cpihhXpf1QyBt',
    'SH_SY5Y_3':'12_6fz_2g0OBROfBxi3IsvBkg9Q8AuKmO',
    'Input':'1F58gLBc_48r_R2dkRVc8ILvnNJW8Lt1n',
}
OUT=Path('/content/drive/MyDrive/Genixcure_PCR_Merge_Analysis/All_10_Samples')
TMP=Path('/content/pcr_merge_work')
OUT.mkdir(parents=True,exist_ok=True); TMP.mkdir(exist_ok=True)
SORT_RAM='1G'  # sort에 허용할 RAM. 고용량 RAM이면 증액 가능
OVERWRITE=False
USE_FIFO=True  # Avoid merged FASTQ write and read
FASTP_THREADS=4
print('Output:',OUT)


In [ ]:
def list_children(folder_id):
    items=[]; token=None
    while True:
        response=service.files().list(
            q=f"'{folder_id}' in parents and trashed=false",
            fields='nextPageToken,files(id,name,mimeType,size,md5Checksum)',
            pageSize=1000,pageToken=token,supportsAllDrives=True,
            includeItemsFromAllDrives=True).execute()
        items.extend(response.get('files',[]))
        token=response.get('nextPageToken')
        if not token: return items

def discover_pair(sample,folder_id):
    items=list_children(folder_id)
    r1=[x for x in items if re.search(r'_1\.(?:fq|fastq)\.gz$',x['name'],re.I)]
    r2=[x for x in items if re.search(r'_2\.(?:fq|fastq)\.gz$',x['name'],re.I)]
    if len(r1)!=1 or len(r2)!=1:
        return None, f'FASTQ pairs unavailable: R1={len(r1)}, R2={len(r2)}'
    base=lambda x: re.sub(r'_[12]\.(?:fq|fastq)\.gz$','',x['name'],flags=re.I)
    if base(r1[0])!=base(r2[0]):
        return None, 'R1/R2 filenames do not form a matching pair'
    return (r1[0],r2[0]),'READY'

inventory=[]; pairs={}
for sample,fid in FOLDER_IDS.items():
    try:
        pair,status=discover_pair(sample,fid)
        if pair: pairs[sample]=pair
    except Exception as e:
        status='ACCESS ERROR: '+str(e)[:180]
    inventory.append({'sample':sample,'status':status})
inv=pd.DataFrame(inventory)
display(inv)
inv.to_csv(OUT/'sample_fastq_inventory.csv',index=False)
print('Ready samples:',len(pairs),'of',len(FOLDER_IDS))


## Execute: Drive API download → fastp merge → FIFO streaming unique count → compressed CSV

Every successful sample is validated with a sum of read counts, length check, and comparison to fastp merged pair count. Only successful files are committed to the output directory. Failed samples preserve their downloaded FASTQ for retry.


In [ ]:
def download_item(item, target_dir):
    dest=target_dir/item['name']
    size=int(item.get('size') or 0)
    if dest.exists() and dest.stat().st_size!=size: dest.unlink()
    if not dest.exists():
        # Per-thread authorized client avoids sharing one non-thread-safe httplib2 connection.
        import google.auth
        credentials,_=google.auth.default(scopes=['https://www.googleapis.com/auth/drive.readonly'])
        client=build('drive','v3',credentials=credentials,cache_discovery=False)
        request=client.files().get_media(fileId=item['id'],supportsAllDrives=True)
        with open(dest,'wb') as f:
            loader=MediaIoBaseDownload(f,request,chunksize=16*1024*1024)
            done=False
            while not done: _,done=loader.next_chunk(num_retries=5)
    if size and dest.stat().st_size!=size: raise RuntimeError('size mismatch: '+dest.name)
    expected=item.get('md5Checksum')
    if expected:
        h=hashlib.md5()
        with open(dest,'rb') as f:
            for block in iter(lambda:f.read(8*1024*1024),b''): h.update(block)
        if h.hexdigest()!=expected: raise RuntimeError('MD5 mismatch: '+dest.name)
    return dest

import shlex
import threading

def unique_shell():
    # Read FASTQ sequences, sort and count identical exact strings. Output gzip CSV in one pass.
    # LC_ALL=C gives bytewise deterministic ordering; no Python RAM-sized Counter.
    return r"""set -o pipefail
mkdir -p "$3"
gzip -dc "$1" | awk 'NR%4==2 {print}' | LC_ALL=C sort --parallel=4 -S "$4" -T "$3" | LC_ALL=C uniq -c | awk 'BEGIN {print "sequence,read_count,length_bp"} {n=$1; $1=""; sub(/^ /, ""); print $0 "," n "," length($0)}' | gzip -1 > "$2"
"""

def count_merged_stream(merged_path, output_path, sort_tmp):
    script=unique_shell()
    subprocess.run(['bash','-n','-c',script],check=True)
    proc=subprocess.run(['bash','-c',script,'bash',str(merged_path),str(output_path),str(sort_tmp),SORT_RAM],capture_output=True,text=True)
    if proc.returncode:
        raise RuntimeError(f'Counting exit={proc.returncode}: {proc.stderr[-2500:]}')

def do_fastp(sample, d, work, local_csv):
    merged=work/('merged.fq.gz' if USE_FIFO else 'merged_file.fq.gz')
    if merged.exists(): merged.unlink()
    if USE_FIFO: os.mkfifo(merged)
    cmd=['fastp','-i',str(d[0]),'-I',str(d[1]),'--merge',
        '--merged_out',str(merged),'--disable_adapter_trimming',
        '--disable_quality_filtering','--disable_length_filtering',
        '--disable_trim_poly_g','--overlap_len_require','20',
        '--overlap_diff_limit','5','--overlap_diff_percent_limit','20',
        '--thread',str(FASTP_THREADS),'--json',str(work/'fastp.json'),
        '--html',str(work/'fastp.html')]
    # Distinct output paths required by fastp; stdout is redirected to DEVNULL to discard R2 without disk I/O.
    cmd.extend(['-o','/dev/null','-O','/dev/stdout'])
    if USE_FIFO:
        # Start the reader first; the producer then opens the FIFO and streams its gzip output.
        errors=[]
        def consumer():
            try: count_merged_stream(merged,local_csv,work/'sort_tmp')
            except Exception as ex: errors.append(ex)
        thread=threading.Thread(target=consumer,daemon=True)
        thread.start()
        proc=subprocess.run(cmd,stdout=subprocess.DEVNULL,stderr=subprocess.PIPE,text=True)
        if proc.returncode:
            # If producer failed before opening fifo, wake blocked reader to let it exit.
            if thread.is_alive():
                try:
                    fd=os.open(str(merged),os.O_WRONLY|os.O_NONBLOCK)
                    os.close(fd)
                except OSError: pass
        thread.join(timeout=60)
        if thread.is_alive(): raise RuntimeError('Unique pipeline did not finish after fastp; inspect disk/RAM')
        if proc.returncode: raise RuntimeError(f'fastp exit={proc.returncode}: {proc.stderr[-2500:]}')
        if errors: raise errors[0]
        merged.unlink(missing_ok=True)
    else:
        proc=subprocess.run(cmd,stdout=subprocess.DEVNULL,stderr=subprocess.PIPE,text=True)
        if proc.returncode: raise RuntimeError(f'fastp exit={proc.returncode}: {proc.stderr[-2500:]}')
        count_merged_stream(merged,local_csv,work/'sort_tmp')
        merged.unlink(missing_ok=True)

def inspect_result(path):
    unique_n=0; merged_by_counts=0
    with gzip.open(path,'rt') as f:
        if f.readline().strip()!='sequence,read_count,length_bp':
            raise ValueError('Bad CSV header')
        for line in f:
            seq,count,length=line.rstrip('\n').split(',')
            if len(seq)!=int(length) or not seq: raise ValueError('Invalid sequence length')
            if int(count)<=0: raise ValueError('Nonpositive count')
            unique_n+=1; merged_by_counts+=int(count)
    return unique_n,merged_by_counts

rows=[]
for sample in FOLDER_IDS:
    output=OUT/f'{sample}_merged_unique_sequences.csv.gz'
    if output.exists() and not OVERWRITE:
        try:
            unique_n,merged_n=inspect_result(output)
            print(f'[SKIP] {sample}: verified existing output, merged={merged_n:,}, unique={unique_n:,}')
            rows.append({'sample':sample,'status':'EXISTS VERIFIED','merged_reads':merged_n,'unique_sequences':unique_n})
            continue
        except Exception as e:
            print(f'[{sample}] Existing output invalid, reprocessing: {e}')
    if sample not in pairs:
        rows.append({'sample':sample,'status':'MISSING FASTQ'})
        continue
    pair=pairs[sample]
    work=TMP/sample
    work.mkdir(parents=True,exist_ok=True)
    local_csv=work/(sample+'_merged_unique_sequences.csv.gz')
    try:
        t=time.time()
        print(f'[{sample}] Downloading R1/R2 in parallel...')
        with ThreadPoolExecutor(max_workers=2) as executor:
            d=list(executor.map(lambda x:download_item(x,work),pair))
        print(f'[{sample}] fastp overlap merge and exact-sequence counting...')
        do_fastp(sample,d,work,local_csv)
        unique_n,merged_n=inspect_result(local_csv)
        report=json.loads((work/'fastp.json').read_text())
        before=int(report['summary']['before_filtering']['total_reads'])//2
        merge_report=report.get('merged_and_filtered',{}).get('total_reads')
        if merge_report is not None and int(merge_report)!=merged_n:
            raise ValueError(f'fastp merged={merge_report} differs from unique count sum={merged_n}')
        if merged_n>before: raise ValueError('merged reads exceed total read pairs')
        # Copy to a staging name on Drive, then rename to final upon successful copy.
        staged=output.with_name(output.name+'.partial')
        shutil.copyfile(local_csv,staged)
        inspect_result(staged)
        staged.replace(output)
        elapsed=round(time.time()-t,1)
        rows.append({'sample':sample,'status':'DONE','total_read_pairs':before,
                     'merged_reads':merged_n,'unique_sequences':unique_n,
                     'merge_rate_pct':round(100*merged_n/max(before,1),3), 'elapsed_sec':elapsed})
        print(f'[{sample}] DONE merged={merged_n:,}, unique={unique_n:,}, {elapsed}s')
    except Exception as exc:
        rows.append({'sample':sample,'status':'ERROR: '+str(exc)[:200]})
        print(f'[{sample}] ERROR: {exc}')
        output.with_name(output.name+'.partial').unlink(missing_ok=True)
    finally:
        if rows[-1]['status']=='DONE': shutil.rmtree(work,ignore_errors=True)
        else: print(f'[{sample}] Retained local files for retry: {work}')
    pd.DataFrame(rows).to_csv(OUT/'batch_merge_summary.csv',index=False)
summary=pd.DataFrame(rows)
display(summary)
summary.to_csv(OUT/'batch_merge_summary.csv',index=False)
print('Outputs:',OUT)


## Deliverables

- Each completed sample: `<sample>_merged_unique_sequences.csv.gz` (the only per-sample result)
- `batch_merge_summary.csv` and `sample_fastq_inventory.csv` to track completeness

To retry only failures, execute again (`OVERWRITE=False`). If you need to preserve *exact* results from an earlier run, leave `OVERWRITE=False`. This notebook preserves the fastp parameters but uses a FIFO for intermediate output; validate with a representative sample before replacing finalized results.
